In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────
SHAPE_LOG       = r"C:/ClaudeCode/research/01. regime_identification/input/shape_log.csv"
ENRICHED_LOG    = r"C:/ClaudeCode/research/03. validation_analysis/shape_log_enriched.csv"
STOCK           = r"C:/ClaudeCode/Raw Data/Stock and Production/FCPO Stock 3Y.xlsx"
PRODUCTION      = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Production 3Y.xlsx"
EXPORT          = r"C:/ClaudeCode/Raw Data/Stock and Production/MPOB Export 3Y.xlsx"
USDMYR          = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/FX_IDC_USDMYR_1D_1227e.xlsx"
PALMSOY         = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/MYX_DLY_FCPO1!_2_CBOT_DL_ZL1!_1D_6.xlsx"
CRUDE           = r"C:/ClaudeCode/Raw Data/Variable Analysis Extra Data/NYMEX_DL_CL1!_1D_84001.xlsx"
ENSO_ASCII      = r"C:/ClaudeCode/Raw Data/ENSO/oni.ascii.txt"
LOG_FILE        = r"C:/ClaudeCode/research/03. validation_analysis/further_validation_log.txt"

# ── shape name mapping ─────────────────────────────────────────
SHAPE_NAMES = {
    '0.0': 'SB  (Steep Backwardation)',
    '0.1': 'MB  (Mild Backwardation)',
    '0.2': 'TB  (Transitional Backwardation)',
    '1':   'C   (Contango)',
    '2':   'F   (Flat/Back-Inversion)',
}
SHAPE_SHORT = {
    '0.0': 'SB', '0.1': 'MB',
    '0.2': 'TB', '1': 'C', '2': 'F',
}

def display_shape(code):
    return SHAPE_NAMES.get(str(code), str(code))

def display_short(code):
    return SHAPE_SHORT.get(str(code), str(code))

def write_log(study_id, content):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    header = f"""
{"="*70}
{study_id}
Run: {timestamp}
{"="*70}
"""
    with open(LOG_FILE, 'a') as f:
        f.write(header + content + "\n")
    print(f"[LOG] Written — {study_id}")

TRAIN_END  = '2024-12-31'
TEST_START = '2025-01-01'

print("Setup complete.")
print("Shape name mapping:")
for k, v in SHAPE_NAMES.items():
    print(f"  {k} -> {v}")

In [ ]:
# ── load shape log (enriched version with duration + prior) ───
df_enriched = pd.read_csv(
    ENRICHED_LOG,
    dtype={'shape': str, 'prior_shape': str},
    parse_dates=['date'])
df_enriched = df_enriched[
    df_enriched['date'] >= '2017-01-01'
].sort_values('date').reset_index(drop=True)

# ── load MPOB ──────────────────────────────────────────────────
def load_mpob(path, col_name):
    df = pd.read_excel(path, parse_dates=True)
    date_col = df.columns[0]
    val_col  = df.columns[1]
    df = df.rename(columns={date_col: 'date', val_col: col_name})
    df['date'] = pd.to_datetime(df['date'])
    return df[['date', col_name]].sort_values('date')

stock = load_mpob(STOCK,      'stock_raw')
prod  = load_mpob(PRODUCTION, 'production_raw')
exp   = load_mpob(EXPORT,     'export_raw')

# ── load macro ─────────────────────────────────────────────────
def load_macro(path, col_name):
    df = pd.read_excel(path, parse_dates=True)
    date_col = df.columns[0]
    val_col  = df.columns[1]
    df = df.rename(columns={date_col: 'date', val_col: col_name})
    df['date'] = pd.to_datetime(df['date'])
    return df[['date', col_name]].sort_values('date')

usdmyr  = load_macro(USDMYR,   'usd_myr')
palmsoy = load_macro(PALMSOY,  'palm_soy')
crude   = load_macro(CRUDE,    'crude_oil')

# ── load ENSO ──────────────────────────────────────────────────
enso = pd.read_csv(ENSO_ASCII, sep=r'\s+',
    names=['year','jan','feb','mar','apr','may',
           'jun','jul','aug','sep','oct','nov','dec'])
enso_long = enso.melt(
    id_vars='year', var_name='month', value_name='oni')
month_map = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,
             'jun':6,'jul':7,'aug':8,'sep':9,
             'oct':10,'nov':11,'dec':12}
enso_long['month'] = enso_long['month'].map(month_map)
enso_long['date']  = pd.to_datetime(
    enso_long[['year','month']].assign(day=1))
enso_long = enso_long[['date','oni']].dropna()

# ── merge everything onto enriched shape log ───────────────────
df = df_enriched.copy()

for src, col in [(stock,'stock_raw'),
                  (prod, 'production_raw'),
                  (exp,  'export_raw')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

df = pd.merge_asof(
    df.sort_values('date'),
    enso_long.sort_values('date'),
    on='date', direction='backward')

for src, col in [(usdmyr,'usd_myr'),
                  (palmsoy,'palm_soy'),
                  (crude,  'crude_oil')]:
    df = pd.merge_asof(
        df.sort_values('date'),
        src.sort_values('date'),
        on='date', direction='backward')

# ── derived variables ──────────────────────────────────────────
df['stock_pct']        = df['stock_raw'].rank(pct=True)
df['prod_mom_1m']      = df['production_raw'].pct_change(21)*100
df['prod_mom_3m']      = df['production_raw'].pct_change(63)*100
df['prod_yoy']         = df['production_raw'].pct_change(252)*100
df['export_yoy']       = df['export_raw'].pct_change(252)*100
df['usd_myr_chg_4w']   = df['usd_myr'].pct_change(20)*100
df['palm_soy_chg_4w']  = df['palm_soy'].pct_change(20)*100
df['crude_chg_4w']     = df['crude_oil'].pct_change(20)*100
df['month']            = df['date'].dt.month

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['prior_shape_enc'] = le.fit_transform(
    df['prior_shape'].fillna('unknown'))

df['stock_state'] = np.where(
    df['stock_pct'] >= df['stock_pct'].median(),
    'High', 'Low')
df['prod_state'] = np.where(
    df['prod_mom_3m'] >= 0, 'Rising', 'Falling')
df['stock_prod_quad'] = df['stock_state'] + '_' + df['prod_state']

quad_map = {
    'High_Falling': 3,
    'Low_Rising':   2,
    'Low_Falling':  1,
    'High_Rising':  0,
}
df['stock_prod_interaction'] = df['stock_prod_quad'].map(
    quad_map).fillna(0)

print(f"Panel built: {len(df)} rows")
print(f"Date range: {df['date'].min().date()} to "
      f"{df['date'].max().date()}")
print(f"\nMissing values in key columns:")
key_cols = ['stock_pct','prod_mom_3m','oni',
            'usd_myr_chg_4w','palm_soy_chg_4w',
            'prior_shape_enc','days_in_shape']
print(df[key_cols].isnull().sum())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import numpy as np

# ── feature set ───────────────────────────────────────────────
FEATURES = [
    'stock_pct', 'prod_mom_3m', 'prod_yoy',
    'export_yoy', 'oni', 'usd_myr_chg_4w',
    'palm_soy_chg_4w', 'days_in_shape',
    'prior_shape_enc', 'month',
    'stock_prod_interaction',
]

# ── build targets ──────────────────────────────────────────────
df_tm = df.sort_values('date').reset_index(drop=True)

# 1-week target (+5 trading days)
df_tm['target_1w'] = df_tm['shape'].shift(-5)

# 2-week target (+14 calendar days, ~10 trading days)
df_tm['target_2w'] = df_tm['shape'].shift(-10)

# Drop rows without both targets
df_tm = df_tm.dropna(subset=FEATURES + ['target_1w','target_2w'])

# Train/test split
df_train = df_tm[df_tm['date'] <= TRAIN_END].copy()
df_test  = df_tm[df_tm['date'] >= TEST_START].copy()

print(f"Train: {len(df_train)} rows")
print(f"Test:  {len(df_test)} rows")

# ── fit transition models (same as Notebook 2) ────────────────
def fit_transition_model(current_shape, df_train,
                          df_test, features, target_col,
                          horizon_label):
    tr = df_train[df_train['shape'] == current_shape
                  ].dropna(subset=features+[target_col])
    te = df_test[ df_test['shape']  == current_shape
                  ].dropna(subset=features+[target_col])

    result = {
        'shape': current_shape,
        'horizon': horizon_label,
        'train_n': len(tr),
        'test_n':  len(te),
    }

    if len(tr) < 30:
        result['status'] = 'INSUFFICIENT DATA'
        return result

    X_tr = tr[features].values
    y_tr = tr[target_col].values
    X_te = te[features].values if len(te) >= 5 else None
    y_te = te[target_col].values if len(te) >= 5 else None

    from collections import Counter
    baseline_shape = Counter(y_tr).most_common(1)[0][0]
    baseline_acc   = (y_tr == baseline_shape).mean()*100

    trans_dist = (pd.Series(y_tr)
                    .value_counts(normalize=True)*100
                    ).round(1).to_dict()

    model = RandomForestClassifier(
        n_estimators=300, max_depth=5,
        min_samples_leaf=8, random_state=42,
        class_weight='balanced')

    cv = cross_val_score(
        model, X_tr, y_tr,
        cv=StratifiedKFold(min(5, len(tr)//10)),
        scoring='accuracy').mean()*100

    model.fit(X_tr, y_tr)

    result['baseline_acc'] = round(baseline_acc, 1)
    result['trans_dist']   = trans_dist
    result['cv_acc']       = round(cv, 1)
    result['model']        = model
    result['classes']      = list(model.classes_)
    result['features']     = features
    result['feature_imp']  = dict(zip(
        features, model.feature_importances_.round(3)))

    if X_te is not None:
        y_pred   = model.predict(X_te)
        test_acc = accuracy_score(y_te, y_pred)*100
        result['test_acc'] = round(test_acc, 1)
        result['vs_base']  = round(test_acc - baseline_acc, 1)
        result['y_te']     = y_te
        result['X_te']     = X_te
        result['status']   = 'FITTED'
    else:
        result['test_acc'] = None
        result['status']   = 'FITTED (no test data)'

    return result

shapes = sorted(df_tm['shape'].unique())
results_1w = {}
results_2w = {}

print("Fitting 1-week models...")
for s in shapes:
    results_1w[s] = fit_transition_model(
        s, df_train, df_test, FEATURES,
        'target_1w', '1-week')
    print(f"  {display_short(s)}: "
          f"{results_1w[s].get('status','?')}")

print("\nFitting 2-week models...")
for s in shapes:
    results_2w[s] = fit_transition_model(
        s, df_train, df_test, FEATURES,
        'target_2w', '2-week')
    print(f"  {display_short(s)}: "
          f"{results_2w[s].get('status','?')}")

print("\nTransition models rebuilt for threshold testing.")

In [ ]:
def threshold_test(model, X_te, y_te, classes,
                   thresholds=[0.05, 0.08, 0.10, 0.15],
                   current_shape='?', horizon='?'):
    """
    For each probability threshold, finds transitions
    where the model assigns probability BELOW that
    threshold and checks the actual occurrence rate.
    A good threshold shows near-zero actual rate —
    meaning the transition can be safely ruled out.
    """
    probs  = model.predict_proba(X_te)
    prob_df = pd.DataFrame(probs, columns=classes)
    prob_df['actual'] = list(y_te)

    results = []
    for next_s in classes:
        sub = prob_df[[next_s,'actual']].copy()
        sub['occurred'] = (sub['actual']==next_s).astype(int)
        base_rate = sub['occurred'].mean()*100

        for thresh in thresholds:
            below = sub[sub[next_s] < thresh]
            if len(below) < 5:
                continue
            actual_rate = below['occurred'].mean()*100
            safe = actual_rate < (base_rate * 0.5)
            results.append({
                'current':   current_shape,
                'next':      next_s,
                'horizon':   horizon,
                'threshold': thresh,
                'n_below':   len(below),
                'base_rate': round(base_rate, 1),
                'actual_rate': round(actual_rate, 1),
                'reduction': round(
                    base_rate - actual_rate, 1),
                'safe_to_ignore': safe,
            })
    return results

print("Threshold test function ready.")

In [ ]:
THRESHOLDS = [0.05, 0.08, 0.10, 0.15]
all_thresh_1w = []
all_thresh_2w = []

for horizon_label, results_dict, store in [
    ('1-week', results_1w, all_thresh_1w),
    ('2-week', results_2w, all_thresh_2w),
]:
    output_lines = [
        f"THRESHOLD TEST — {horizon_label.upper()}",
        "="*70,
        "Shows: when model probability < threshold,",
        "       how often does that transition still occur?",
        "SAFE TO IGNORE = actual rate < 50% of base rate",
    ]
    print("\n".join(output_lines))

    for s in shapes:
        r = results_dict[s]
        if r.get('status') != 'FITTED' or 'model' not in r:
            continue
        if r.get('X_te') is None:
            continue

        thresh_results = threshold_test(
            r['model'], r['X_te'], r['y_te'],
            r['classes'], THRESHOLDS, s, horizon_label)
        store.extend(thresh_results)

        line = f"\n  {display_shape(s)}:"
        print(line); output_lines.append(line)

        line = (f"  {'Next':<6} {'Thresh':>7} "
                f"{'n<thresh':>9} {'Base%':>7} "
                f"{'Actual%':>8} {'Reduction':>10} "
                f"{'Safe?':>8}")
        print(line); output_lines.append(line)

        for tr in thresh_results:
            flag = " SAFE" if tr['safe_to_ignore'] else ""
            line = (f"    {display_short(tr['next']):<4} "
                    f"{tr['threshold']:>6.0%} "
                    f"{tr['n_below']:>9} "
                    f"{tr['base_rate']:>6.1f}% "
                    f"{tr['actual_rate']:>7.1f}% "
                    f"{tr['reduction']:>+9.1f}pp"
                    f"{flag}")
            print(line); output_lines.append(line)

    write_log(
        f"FURTHER STUDY 3 — THRESHOLD TEST "
        f"{horizon_label.upper()}",
        "\n".join(output_lines))

In [ ]:
output_lines = [
    "THRESHOLD TEST — RULING OUT TABLE",
    "The practical output: which transitions can be",
    "safely ignored at which probability threshold",
    "across both horizons?",
    "="*70,
]
print("\n".join(output_lines))

# Find lowest threshold where safe_to_ignore = True
# for each (current, next, horizon) combination
safe_table = {}
for store, horizon in [(all_thresh_1w, '1-week'),
                        (all_thresh_2w, '2-week')]:
    for tr in store:
        key = (tr['current'], tr['next'], horizon)
        if tr['safe_to_ignore']:
            if key not in safe_table:
                safe_table[key] = tr
            elif tr['threshold'] < safe_table[key]['threshold']:
                safe_table[key] = tr

print(f"\n{'Current':<6} {'Next':<6} {'Horizon':<8} "
      f"{'Threshold':>10} {'Base%':>7} "
      f"{'Below thresh%':>14} {'Reduction':>10}")
print("-"*65)
output_lines.append(
    f"\n{'Current':<6} {'Next':<6} {'Horizon':<8} "
    f"{'Threshold':>10} {'Base%':>7} "
    f"{'Below thresh%':>14} {'Reduction':>10}")
output_lines.append("-"*65)

for (curr, next_s, hor), tr in sorted(
        safe_table.items(),
        key=lambda x: (x[0][0], x[0][2], -x[1]['reduction'])):
    line = (f"  {display_short(curr):<4} "
            f"-> {display_short(next_s):<4} "
            f"{hor:<8} "
            f"< {tr['threshold']:.0%}     "
            f"{tr['base_rate']:>6.1f}%  "
            f"{tr['actual_rate']:>12.1f}%  "
            f"{tr['reduction']:>+9.1f}pp")
    print(line); output_lines.append(line)

# Summary by shape
print("\nSUMMARY — TRANSITIONS THAT CAN BE RULED OUT "
      "PER CURRENT SHAPE:")
output_lines.append(
    "\nSUMMARY — TRANSITIONS THAT CAN BE RULED OUT:")
for s in shapes:
    safe_1w = [k for k in safe_table
               if k[0]==s and k[2]=='1-week']
    safe_2w = [k for k in safe_table
               if k[0]==s and k[2]=='2-week']
    line = (f"\n  {display_shape(s)}:\n"
            f"    1-week ruled out: "
            f"{[display_short(k[1]) for k in safe_1w] or 'none'}\n"
            f"    2-week ruled out: "
            f"{[display_short(k[1]) for k in safe_2w] or 'none'}")
    print(line); output_lines.append(line)

write_log("FURTHER STUDY 3 — RULING OUT TABLE",
          "\n".join(output_lines))
print("\n[NOTEBOOK 3 COMPLETE]")